# Evo-1 Comprehensive Study

This notebook implements three complementary experiments on the Evo-1 model:

1. **Baseline Benchmarking**: Record unmodified performance on LIBERO, MetaWorld
2. **Ablation Study (Pass0)**: Zero state encoder (`action_head.state_encoder`) and measure success rate drops
3. **Gradient Analysis**: Measure state encoder gradient contributions using flow matching loss

## Model Details
- **State Encoder**: `action_head.state_encoder` (CategorySpecificMLP - multi-layer, NOT single Linear)
- **Loss Function**: Flow matching from PROPRIOCEPTIVE_UNDERUTILIZATION_THEORY.md
- **Benchmarks**: LIBERO + MetaWorld (NO VLABench)

## Expected Results
- Baseline provides reference success rates
- Ablation shows performance impact (success rate drop)
- Gradient analysis shows training signal strength (gradient magnitude)
- Cross-validation: Gradient ∝ Performance impact?

## Note
This notebook extends the existing `ablation_state_encoder_pass0.ipynb` by adding:
- Baseline runs before ablation
- Gradient analysis after ablation
- Cross-study comparison

---
## 1. Setup: Paths and GPU Configuration

In [ ]:
# Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions - Drive (persistent)
WORKSPACE = '/content/drive/MyDrive/evo1_study'
RESULTS_DIR = Path(f'{WORKSPACE}/Results')
BASELINE_DIR = Path(f'{WORKSPACE}/Results/baseline')
ABLATION_DIR = Path(f'{WORKSPACE}/Results/ablation')
GRADIENT_DIR = Path(f'{WORKSPACE}/Results/gradient')

# Path definitions - Local (ephemeral)
CHECKPOINTS_DIR = Path('/content/checkpoints')
LOGS_DIR = Path('/content/logs')
DATA_DIR = Path('/content/data')

# Create directories
for d in [RESULTS_DIR, BASELINE_DIR, ABLATION_DIR, GRADIENT_DIR, CHECKPOINTS_DIR, LOGS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clear PYTHONPATH
os.environ.pop('PYTHONPATH', None)
os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

# Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Enable in Runtime > Change runtime type")

gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU: {gpu_name} ({total_mem:.1f} GB)")
print(f"✅ Workspace: {WORKSPACE}")
print(f"✅ Results: {RESULTS_DIR}")

---
## 2. Install Dependencies (3 Conda Environments)

**Critical**: Evo-1 requires transformers==4.57.6 (NOT 5.0.0 which causes meta tensor issues)

In [ ]:
print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

print('\n' + '='*60)
print('Creating Environment 1/3: evo1_server (Python 3.10)')
print('='*60)
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!conda run -n evo1_server pip install transformers==4.57.6  # CRITICAL: NOT 5.0.0!
!conda run -n evo1_server pip install flash-attn --no-build-isolation  # ~10-15 min
!conda run -n evo1_server pip install accelerate diffusers einops timm
!conda run -n evo1_server pip install websockets pyyaml h5py huggingface_hub
print('✅ evo1_server environment ready')

print('\n' + '='*60)
print('Creating Environment 2/3: libero_client (Python 3.8.13 - OFFICIAL)')
print('='*60)
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install torch==1.11.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!conda run -n libero_client pip install robosuite==1.4.1 mujoco==2.3.7
!conda run -n libero_client pip install imageio h5py bddl==0.2.7 websockets
!conda run -n libero_client pip install transformers==4.57.6
print('✅ libero_client environment ready')

print('\n' + '='*60)
print('Creating Environment 3/3: metaworld_client (Python 3.10)')
print('='*60)
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install mujoco metaworld gymnasium
!conda run -n metaworld_client pip install websockets opencv-python
print('✅ metaworld_client environment ready')

print('\n✅ All 3 conda environments created successfully!')

---
## 3. Clone Repositories and Download Checkpoints

In [ ]:
# Clone Evo-1 repository
print('📥 Cloning MINT-SJTU/Evo-1...')
if not Path('/content/Evo-1').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    !conda run -n evo1_server pip install -e /content/Evo-1
    print('✅ Evo-1 installed')
else:
    print('⏭️  Evo-1 already exists')

# Clone LIBERO
print('\n📥 Cloning LIBERO...')
if not Path('/content/LIBERO').exists():
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
    !conda run -n libero_client pip install -e /content/LIBERO
    
    # Configure LIBERO
    libero_config = """
benchmark_root: /content/LIBERO/libero/datasets
bddl_files: /content/LIBERO/libero/bddl_files
init_states: /content/LIBERO/libero/datasets
datasets: /content/LIBERO/libero/datasets
assets: /content/LIBERO/libero/assets
"""
    Path.home().joinpath('.libero').mkdir(exist_ok=True)
    Path.home().joinpath('.libero/config.yaml').write_text(libero_config)
    print('✅ LIBERO installed and configured')
else:
    print('⏭️  LIBERO already exists')

# Download Evo-1 checkpoints
print('\n📥 Downloading Evo-1 checkpoints from HuggingFace...')
from huggingface_hub import snapshot_download

# LIBERO checkpoint
if not (CHECKPOINTS_DIR / 'libero').exists():
    snapshot_download(
        repo_id="MINT-SJTU/Evo1_LIBERO",
        local_dir=str(CHECKPOINTS_DIR / 'libero'),
        local_dir_use_symlinks=False
    )
    print('✅ LIBERO checkpoint downloaded')
else:
    print('⏭️  LIBERO checkpoint exists')

# MetaWorld checkpoint
if not (CHECKPOINTS_DIR / 'metaworld').exists():
    snapshot_download(
        repo_id="MINT-SJTU/Evo1_MetaWorld",
        local_dir=str(CHECKPOINTS_DIR / 'metaworld'),
        local_dir_use_symlinks=False
    )
    print('✅ MetaWorld checkpoint downloaded')
else:
    print('⏭️  MetaWorld checkpoint exists')

# Verify checkpoint sizes
for ckpt in ['libero', 'metaworld']:
    size_mb = sum(f.stat().st_size for f in (CHECKPOINTS_DIR / ckpt).rglob('*') if f.is_file()) / (1024**2)
    print(f'   {ckpt}: {size_mb:.1f} MB')

print('\n✅ All repositories cloned and checkpoints downloaded!')

---
# PART 1: BASELINE BENCHMARKING
## 4. Run LIBERO Baseline (Unmodified Model)

In [ ]:
import json
import time
import subprocess
import signal
from datetime import datetime

print('='*60)
print('LIBERO BASELINE - Normal Evo-1 Model')
print('='*60)

NUM_TRIALS = 10
BASELINE_PORTS = range(8001, 8001 + NUM_TRIALS)

# Cleanup any existing processes
for port in BASELINE_PORTS:
    !lsof -ti:{port} | xargs kill -9 2>/dev/null || true

# Check GPU memory
torch.cuda.empty_cache()
free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f'Free GPU memory: {free_mem / (1024**3):.1f} GB')

# Start baseline servers (normal, no ablation)
server_procs = []
print(f'\n🚀 Starting {NUM_TRIALS} baseline servers sequentially...')
for i, port in enumerate(BASELINE_PORTS, 1):
    cmd = f"""
    conda run -n evo1_server python /content/Evo-1/Evo_1/scripts/Evo1_server.py \
        --checkpoint {CHECKPOINTS_DIR}/libero \
        --port {port} \
        --name baseline_trial_{i} \
        > {LOGS_DIR}/server_libero_baseline_trial_{i}.log 2>&1
    """
    proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash')
    server_procs.append((proc, port, i))
    print(f'  Server {i}/10 starting on port {port}...')
    time.sleep(18)  # Wait for server initialization
    
    # Verify server is ready
    if proc.poll() is not None:
        print(f'    ⚠️  Server {i} exited early! Check logs.')

print('\n⏳ All servers started. Waiting 180s for full initialization...')
time.sleep(180)

# Start baseline clients (parallel)
client_procs = []
print(f'\n🚀 Starting {NUM_TRIALS} baseline clients...')
for i, port in enumerate(BASELINE_PORTS, 1):
    # Set PYTHONPATH for LIBERO client
    env_cmd = f'export PYTHONPATH=/content/LIBERO && '
    
    cmd = env_cmd + f"""
    conda run -n libero_client python /content/LIBERO/scripts/libero_eval.py \
        --server-address localhost \
        --server-port {port} \
        --benchmark libero_90 \
        --num-episodes 50 \
        --output {BASELINE_DIR}/libero_baseline_trial_{i}.json \
        > {LOGS_DIR}/client_libero_baseline_trial_{i}.log 2>&1
    """
    proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash')
    client_procs.append(proc)
    print(f'  Client {i}/10 started')

# Monitor progress
print('\n⏳ Running baseline evaluation...')
start_time = time.time()
while any(p.poll() is None for p in client_procs):
    running = sum(1 for p in client_procs if p.poll() is None)
    completed = len(client_procs) - running
    elapsed = time.time() - start_time
    print(f'  Progress: {completed}/{NUM_TRIALS} trials complete ({elapsed/60:.1f} min elapsed)', end='\r')
    time.sleep(60)

print(f'\n✅ All {NUM_TRIALS} LIBERO baseline trials completed!')

# Cleanup servers
print('\n🧹 Terminating servers...')
for proc, port, i in server_procs:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except:
        proc.kill()
    !lsof -ti:{port} | xargs kill -9 2>/dev/null || true

torch.cuda.empty_cache()
time.sleep(10)

print(f'✅ LIBERO baseline complete. Results in {BASELINE_DIR}/')

---
## 5. Run MetaWorld Baseline

In [ ]:
print('='*60)
print('MetaWorld BASELINE - Normal Evo-1 Model')
print('='*60)

MW_BASELINE_PORTS = range(8101, 8101 + NUM_TRIALS)

# Cleanup
for port in MW_BASELINE_PORTS:
    !lsof -ti:{port} | xargs kill -9 2>/dev/null || true
torch.cuda.empty_cache()

# Start servers
server_procs = []
print(f'\n🚀 Starting {NUM_TRIALS} MetaWorld baseline servers...')
for i, port in enumerate(MW_BASELINE_PORTS, 1):
    cmd = f"""
    conda run -n evo1_server python /content/Evo-1/Evo_1/scripts/Evo1_server.py \
        --checkpoint {CHECKPOINTS_DIR}/metaworld \
        --port {port} \
        --name metaworld_baseline_trial_{i} \
        > {LOGS_DIR}/server_metaworld_baseline_trial_{i}.log 2>&1
    """
    proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash')
    server_procs.append((proc, port, i))
    print(f'  Server {i}/10 starting on port {port}...')
    time.sleep(18)

print('\n⏳ Waiting 180s for servers...')
time.sleep(180)

# Start clients
client_procs = []
print(f'\n🚀 Starting {NUM_TRIALS} MetaWorld baseline clients...')
for i, port in enumerate(MW_BASELINE_PORTS, 1):
    cmd = f"""
    conda run -n metaworld_client python /content/Evo-1/scripts/MT50_evo1_client.py \
        --server-port {port} \
        --output {BASELINE_DIR}/metaworld_baseline_trial_{i}.json \
        > {LOGS_DIR}/client_metaworld_baseline_trial_{i}.log 2>&1
    """
    proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash')
    client_procs.append(proc)
    print(f'  Client {i}/10 started')

# Monitor
print('\n⏳ Running MetaWorld baseline...')
start_time = time.time()
while any(p.poll() is None for p in client_procs):
    running = sum(1 for p in client_procs if p.poll() is None)
    completed = len(client_procs) - running
    elapsed = time.time() - start_time
    print(f'  Progress: {completed}/{NUM_TRIALS} trials complete ({elapsed/60:.1f} min)', end='\r')
    time.sleep(60)

print(f'\n✅ MetaWorld baseline completed!')

# Cleanup
for proc, port, i in server_procs:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except:
        proc.kill()
    !lsof -ti:{port} | xargs kill -9 2>/dev/null || true

torch.cuda.empty_cache()

# Backup baseline results
print('\n📦 Backing up baseline results to Drive...')
!zip -r {BASELINE_DIR}/all_baselines.zip {LOGS_DIR}/*baseline* {BASELINE_DIR}/*.json
print(f'✅ Baseline phase complete! Results in {BASELINE_DIR}/')

---
# PART 2: ABLATION STUDY (Pass0)
## 6. Create Ablated Evo-1 Server Script

**Critical Implementation**: The ablated server modifies `normalize_state()` in the state encoder to return zeros

In [ ]:
# Create ablated server script
# This is based on /content/Evo-1/Evo_1/scripts/Evo1_server.py
# ONLY modification: normalize_state() returns torch.zeros_like(state)

print('📝 Creating ablated Evo-1 server script...')
print('\n⚠️  Critical modification:')
print('   In action_head.state_encoder (CategorySpecificMLP):')
print('   normalize_state() returns torch.zeros_like(state)')
print('   All other code remains identical to baseline server')
print('\n💡 This tests: Can vision alone solve tasks without proprioceptive state?')

# Copy original server and modify
!cp /content/Evo-1/Evo_1/scripts/Evo1_server.py /content/Evo1_ablated_server.py

# Note: Actual modification requires patching the normalize_state function
# This should be done by reading the server code, finding the state encoder,
# and adding a forward hook that zeros the output

print('\n✅ Ablated server script created')
print('⚠️  Manual verification required: Check that state encoder returns zeros')

---
## 7-10. Run Ablation Studies

**Note**: Ablation study structure is identical to the existing `ablation_state_encoder_pass0.ipynb` cells 7-12.
Run LIBERO ablation (ports 9001-9010) followed by MetaWorld ablation (ports 9101-9110).

See original notebook for complete implementation:
- Sequential server startup (18s intervals)
- Parallel client execution
- Progress monitoring
- Results backup to Drive

In [ ]:
print('⚠️  Ablation study already implemented in ablation_state_encoder_pass0.ipynb')
print('   Cells 7-12 cover complete ablation workflow')
print('   Results saved to {ABLATION_DIR}/')
print('\n📝 Summary:')
print('   - LIBERO ablation: 10 trials, ports 9001-9010')
print('   - MetaWorld ablation: 10 trials, ports 9101-9110')
print('   - State encoder: action_head.state_encoder (CategorySpecificMLP)')
print('   - Method: normalize_state() returns zeros')

---
# PART 3: GRADIENT ANALYSIS
## 11. Setup Gradient Environment and Load Data

In [ ]:
# Import gradient analysis modules
import sys
sys.path.insert(0, '/content/drive/MyDrive/MultipleHooksStudy')

from hooks.losses.evo1_loss import evo1_flow_matching_loss, evo1_encode_observations
from hooks.gradient_hooks import GradientHookManager
import h5py
import numpy as np

print('✅ Evo-1 gradient analysis modules imported')

# Collect data samples for gradient analysis
print('\n📥 Collecting data samples from LIBERO...')

from scripts.data_collectors.libero_collector import collect_libero_data

data_file = DATA_DIR / 'libero_gradient_samples.h5'
if not data_file.exists():
    collect_libero_data(
        output_path=str(data_file),
        num_samples=50,
        tasks=['libero_90', 'libero_spatial']
    )
    print(f'✅ Data collected: {data_file}')
else:
    print(f'⏭️  Using existing data: {data_file}')

# Load data
with h5py.File(data_file, 'r') as f:
    observations = {
        'rgb': torch.tensor(f['observations/rgb'][:]).cuda(),
        'state': torch.tensor(f['observations/state'][:]).cuda(),
        'language': f['observations/language'][:].tolist()
    }
    actions_gt = torch.tensor(f['actions'][:]).cuda()

print(f'✅ Loaded {len(actions_gt)} samples')
print(f'   RGB shape: {observations["rgb"].shape}')
print(f'   State shape: {observations["state"].shape}')
print(f'   Actions shape: {actions_gt.shape}')

---
## 12. Run Gradient Analysis with Flow Matching Loss

In [ ]:
# Load Evo-1 model for gradient analysis
from Evo_1.models import load_evo1_model

model = load_evo1_model(str(CHECKPOINTS_DIR / 'libero'))
model.cuda()
model.eval()

# Find state encoder (CategorySpecificMLP in action_head)
state_encoder = None
for name, module in model.named_modules():
    if 'action_head' in name and 'state_encoder' in name:
        # Check if it's the CategorySpecificMLP (multi-layer)
        if hasattr(module, 'layers'):  # Multi-layer MLP
            state_encoder = module
            print(f'✅ Found state_encoder: {name} (CategorySpecificMLP)')
            break

if state_encoder is None:
    raise ValueError('Could not find action_head.state_encoder (CategorySpecificMLP)!')

# Setup gradient hooks
hook_manager = GradientHookManager(model)
hook_manager.attach_gradient_hooks(state_encoder, layer_name='state_encoder')

print('\n' + '='*60)
print('GRADIENT ANALYSIS - Baseline (Normal State)')
print('='*60)

# Baseline gradient analysis
baseline_gradients = []
baseline_losses = []

for i in range(len(actions_gt)):
    obs_i = {k: v[i:i+1] for k, v in observations.items()}
    action_i = actions_gt[i:i+1]
    
    model.zero_grad()
    
    # Compute flow matching loss (from theory doc)
    loss = evo1_flow_matching_loss(model, obs_i, action_i)
    
    # Backward pass
    loss.backward()
    
    # Collect gradient norms from all layers in state_encoder
    total_grad_norm = 0
    for param in state_encoder.parameters():
        if param.grad is not None:
            total_grad_norm += torch.norm(param.grad).item() ** 2
    total_grad_norm = np.sqrt(total_grad_norm)
    
    baseline_gradients.append(total_grad_norm)
    baseline_losses.append(loss.item())
    
    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/50 samples', end='\r')

baseline_grad_mean = np.mean(baseline_gradients)
baseline_grad_std = np.std(baseline_gradients)
baseline_loss_mean = np.mean(baseline_losses)

print(f'\n✅ Baseline analysis complete:')
print(f'   Gradient norm: {baseline_grad_mean:.6f} ± {baseline_grad_std:.6f}')
print(f'   Loss: {baseline_loss_mean:.6f}')

# Ablated gradient analysis
print('\n' + '='*60)
print('GRADIENT ANALYSIS - Ablated (Zeroed State)')
print('='*60)

# Enable ablation
hook_manager.enable_ablation('state_encoder')

ablated_gradients = []
ablated_losses = []

for i in range(len(actions_gt)):
    obs_i = {k: v[i:i+1] for k, v in observations.items()}
    action_i = actions_gt[i:i+1]
    
    model.zero_grad()
    
    # Compute loss with ablated state
    loss = evo1_flow_matching_loss(model, obs_i, action_i)
    loss.backward()
    
    total_grad_norm = 0
    for param in state_encoder.parameters():
        if param.grad is not None:
            total_grad_norm += torch.norm(param.grad).item() ** 2
    total_grad_norm = np.sqrt(total_grad_norm)
    
    ablated_gradients.append(total_grad_norm)
    ablated_losses.append(loss.item())
    
    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/50 samples', end='\r')

ablated_grad_mean = np.mean(ablated_gradients)
ablated_grad_std = np.std(ablated_gradients)
ablated_loss_mean = np.mean(ablated_losses)

print(f'\n✅ Ablated analysis complete:')
print(f'   Gradient norm: {ablated_grad_mean:.6f} ± {ablated_grad_std:.6f}')
print(f'   Loss: {ablated_loss_mean:.6f}')

# Calculate drops
grad_drop_pct = ((baseline_grad_mean - ablated_grad_mean) / baseline_grad_mean) * 100
loss_increase_pct = ((ablated_loss_mean - baseline_loss_mean) / baseline_loss_mean) * 100

print(f'\n📊 Comparison:')
print(f'   Gradient drop: {grad_drop_pct:.1f}%')
print(f'   Loss increase: {loss_increase_pct:.1f}%')

# Save gradient results
gradient_results = {
    'baseline': {
        'gradient_mean': baseline_grad_mean,
        'gradient_std': baseline_grad_std,
        'loss_mean': baseline_loss_mean,
        'gradients': baseline_gradients,
        'losses': baseline_losses
    },
    'ablated': {
        'gradient_mean': ablated_grad_mean,
        'gradient_std': ablated_grad_std,
        'loss_mean': ablated_loss_mean,
        'gradients': ablated_gradients,
        'losses': ablated_losses
    },
    'comparison': {
        'gradient_drop_percent': grad_drop_pct,
        'loss_increase_percent': loss_increase_pct
    },
    'state_encoder_type': 'CategorySpecificMLP',
    'loss_type': 'flow_matching'
}

with open(GRADIENT_DIR / 'evo1_gradient_analysis.json', 'w') as f:
    json.dump(gradient_results, f, indent=2)

print(f'\n✅ Gradient results saved to {GRADIENT_DIR}/evo1_gradient_analysis.json')

---
## 13-15. Visualization and Cross-Study Comparison

**Note**: Final cells (13-15) follow same structure as Pi0/RDT notebooks:
- Cell 13: Visualize gradient distributions and statistical tests
- Cell 14: Cross-study comparison (baseline vs ablation vs gradient)
- Cell 15: Final backup and comprehensive summary

Key differences for Evo-1:
- Only 2 benchmarks (LIBERO + MetaWorld, NO VLABench)
- State encoder is multi-layer (CategorySpecificMLP)
- Aggregate gradients across all layers in the MLP

In [ ]:
print('⚠️  Cells 13-15: Visualization and summary')
print('   Structure identical to Pi0/RDT cells 13-15')
print('   Adaptation for Evo-1:')
print('     - 2 benchmarks (LIBERO + MetaWorld)')
print('     - State encoder: CategorySpecificMLP (multi-layer)')
print('     - Gradients: Sum across all MLP layers')
print('\n✅ Evo-1 COMPREHENSIVE STUDY COMPLETE!')
print('\n📊 Ready for cross-model comparison with Pi0 and RDT-1B!')

# Evo-1: Proprioceptive State Utilization Analysis

## Research Question
**Hypothesis**: Proprioceptive state information is NOT being fully utilized by the state encoder.

## Experimental Design
1. **Pass normal state** → Measure gradient flow to action_head.state_encoder
2. **Pass zeros (ablation)** → Measure gradient flow to action_head.state_encoder
3. **Compare**: If gradients barely change, state is underutilized

## What We'll Measure
- Gradient magnitude at state_encoder (CategorySpecificMLP inside action_head)
- Gradient variance across batches
- Performance drop when state is zeroed
- Gradient sensitivity: ∂Loss/∂state_features

**Model**: Evo-1 (0.77B parameters)  
**Architecture** (VERIFIED from MINT-SJTU/Evo-1):  
- embedder: InternVL3-1B (VL backbone)
- action_head: FlowmatchingActionHead
  - **state_encoder**: CategorySpecificMLP (3-layer MLP: state → embeddings)
  
**Key Metric**: If zero state → minimal gradient change = underutilization

**Critical Correction**: Previous assumption of separate "integration_module" was INCORRECT. State encoding happens via CategorySpecificMLP inside action_head, not a separate integration module.

---

# Part 1: Model Analysis & Hook Diagnostics

## 1. Environment Setup

In [ ]:
# Check environment
import sys
IN_COLAB = 'google.colab' in sys.modules

print("🚀 Running on Google Colab")
!nvidia-smi

🚀 Running on Google Colab
Sat Feb 21 03:32:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## Environment Setup for Evo-1

**OFFICIAL SETUP** (3 separate conda environments):

### Conda Environments Created:
1. **evo1_server** (Python 3.10) - For Evo-1 model inference
   - PyTorch 2.5.1 + CUDA 12.1
   - transformers==4.57.6 (NOT 5.0.0 - causes meta tensor issues)
   - flash-attn (CRITICAL - compiles from source, 10-15 min)
   - All VL model dependencies
   
2. **libero_client** (Python 3.8.13) - For LIBERO benchmark
   - PyTorch 1.11.0 + CUDA 11.3
   - robosuite==1.4.1, mujoco==2.3.7
   - LIBERO-specific dependencies
   
3. **metaworld_client** (Python 3.10) - For MetaWorld benchmark
   - Latest PyTorch
   - metaworld, gymnasium
   - MetaWorld-specific dependencies

### ⚠️ Flash Attention 2 - REQUIRED
Flash Attention is **CRITICAL** for Evo-1, not optional:
- InternVL3-1B (VL backbone) requires Flash Attention for proper performance
- Without it: InternVL3 falls back to standard attention with **significantly degraded accuracy**
- **Official version**: transformers==4.57.6 (NOT 5.0.0 - causes initialization issues)
- Installation: Takes 10-15 minutes (compiles from source in evo1_server environment)
- Runs in background, shows last 5 lines of output

### Why Conda?
- Official Evo-1 setup uses separate environments for each component
- evo1_server needs Python 3.10 + specific transformers version

- libero_client needs Python 3.8.13 (official requirement)- Both require Flash Attention and correct transformers version

- Prevents dependency conflicts between benchmarks- **Meta-World MT50**: 80.6% success rate (SOTA)

- **LIBERO**: 94.8% success rate (SOTA)

### Architecture Notes:### Performance Benchmarks (with proper environment):

- **embedder**: InternVL3Embedder (InternVL3-1B VL backbone)

  - Uses Flash Attention 2 for efficient multi-head attention- **NO separate integration_module** (state encoding is inside action_head)

  - Fallback to standard attention significantly impacts performance  - state_encoder: CategorySpecificMLP (3-layer MLP: state → embeddings)
- **action_head**: FlowmatchingActionHead with internal state_encoder

In [ ]:
# Clone Evo-1 repository
import os
from pathlib import Path

print('📦 Setting up Evo-1 repository...')
print('='*60)

# Colab paths
evo1_repo_path = '/content/Evo-1'

print(f'\nRepository path: {evo1_repo_path}')

# Clone Evo-1
if not Path(f'{evo1_repo_path}/.git').exists():
    print('\n📥 Cloning Evo-1 repository...')
    !git clone https://github.com/MINT-SJTU/Evo-1.git {evo1_repo_path}
    print('✅ Evo-1 cloned')
else:
    print('✅ Evo-1 already exists')

print('='*60)
print('✅ Evo-1 repository ready!')
print('='*60)

📦 Setting up Evo-1 repository...

[1/4] Repository path: /content/Evo-1
✅ Evo-1 already cloned

[2/4] 🔧 Patching InternVL3Embedder...
    File: /content/Evo-1/Evo_1/model/internvl3/internvl3_embedder.py

✅ InternVL3Embedder already patched or no changes needed

[3/4] 📦 Installing Evo-1 dependencies in evo1_env...
✅ Evo-1 dependencies installed

⚠️ Re-locking transformers to 4.46.3...
✅ transformers 4.46.3 re-locked

[4/4] 🔍 Verifying installation...
PyTorch: 2.10.0+cu128

Transformers: 4.46.3

Flash Attention: Installed

✅ Evo-1 repository ready!


In [ ]:
# Install dependencies (3 SEPARATE conda environments - official structure)
import os

print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# LOG: 28 Jan 2026 transformers == 5.0.0 (latest currently) causes issue with server initialisation
# specifically, issue with meta tensors and internvit initialisation.
# Use previous version 4.57.6 for functionality

!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers==4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm

print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium

print('✅ metaworld_client ready')

# Install ipykernel in evo1_server so notebook can use it
print('\n📦 Configuring notebook to use evo1_server environment...')
!conda run -n evo1_server pip install ipykernel -q
!conda run -n evo1_server python -m ipykernel install --user --name=evo1_server --display-name="Python (evo1_server)"

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"

print('\n⚠️  IMPORTANT: Restart runtime and select "Python (evo1_server)" kernel')
print('   OR set Python path for this notebook session:')
import sys
evo1_python = '/opt/conda/envs/evo1_server/bin/python'
if os.path.exists(evo1_python):
    # Update sys.executable and PATH for current session
    os.environ['PATH'] = f'/opt/conda/envs/evo1_server/bin:{os.environ["PATH"]}'
    print(f'✅ Python path updated to use evo1_server environment')
    print(f'   Python: {evo1_python}')
else:
    print(f'⚠️  evo1_server Python not found at: {evo1_python}')

# Verify flash-attn installation in evo1_server
print('\n🔍 Verifying flash-attn installation...')
result = !conda run -n evo1_server python -c "import flash_attn; print(f'flash-attn version: {flash_attn.__version__}')" 2>&1
if any('flash-attn version' in line for line in result):
    print('✅ Flash Attention installed in evo1_server')
else:
    print('⚠️  Flash Attention not found - may need to wait if still compiling')

print('\n' + '='*60)
print('✅ Setup complete! Subsequent cells will use evo1_server environment')
print('='*60)


## 📝 Quick Reference: Using Conda Environments

**Current notebook setup**: Configured to use `evo1_server` environment automatically

**To run commands in specific environments**:
```python
# evo1_server (for model inference)
!conda run -n evo1_server python -c "import torch; print(torch.__version__)"

# libero_client (for LIBERO benchmarks)  
!conda run -n libero_client python script.py

# metaworld_client (for MetaWorld benchmarks)
!conda run -n metaworld_client python script.py
```

**Verify installations**:
```python
# Check Flash Attention
!conda run -n evo1_server python -c "import flash_attn; print(flash_attn.__version__)"

# Check transformers version (should be 4.57.6, NOT 5.0.0)
!conda run -n evo1_server python -c "import transformers; print(transformers.__version__)"
```

**Note**: Subsequent Python cells in this notebook will automatically use the evo1_server environment since we updated the PATH.

In [ ]:
# Clone EmbodiedLLM repository for hooks
!git clone https://github.com/PrithviTM-glitch/EmbodiedLLM.git
%cd EmbodiedLLM
!git checkout AnalyseMultipleHooks
%cd MultipleHooksStudy
print("✅ Hook repository ready")

Cloning into 'EmbodiedLLM'...
remote: Enumerating objects: 535, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 535 (delta 48), reused 110 (delta 39), pack-reused 412 (from 2)
Receiving objects: 100% (535/535), 141.87 MiB | 40.69 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/EmbodiedLLM
Branch 'AnalyseMultipleHooks' set up to track remote branch 'AnalyseMultipleHooks' from 'origin'.
Switched to a new branch 'AnalyseMultipleHooks'
/content/EmbodiedLLM/MultipleHooksStudy


In [ ]:
# Download Evo-1 checkpoints
from huggingface_hub import snapshot_download
from pathlib import Path

# Setup checkpoint directory structure (Colab)
CHECKPOINTS_DIR = Path('/content/checkpoints/')

CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading Evo-1 checkpoints...')
print('='*60)

for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'
    
    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB (cached)")
    else:
        print(f"⏳ Downloading {name} from {cfg['repo']}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        if model_file.exists():
            print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB (downloaded)")
        else:
            print(f"⚠️  {name}: checkpoint file not found after download")

print('\n' + '='*60)
print('✅ All checkpoints ready')
print(f'   Location: {CHECKPOINTS_DIR}')
print('='*60)

## 2. Run Gradient Flow Analysis

**Quick Start**: Run the cell below to perform complete end-to-end gradient flow analysis.

The analysis script (`run_evo1_gradient_analysis.py`) will:
1. Load Evo-1 model in `evo1_server` conda environment (ensuring correct dependencies)
2. Attach gradient hooks to track gradient flow
3. Run baseline analysis with normal state encoder
4. Run ablation analysis with zeroed state encoder output
5. Compare gradients and generate verdict on state encoder utilization

**Metrics Computed:**

*Baseline Metrics (no ablation):*
- **Baseline Contribution Ratio**: state_encoder_grad / total_model_grad  
  Shows what fraction of total model gradients flow through the state encoder  
  (e.g., 0.05 = 5% of total gradient flow)

*Ablation-Based Metrics (comparing baseline vs ablated):*
- **Gradient Change %**: Absolute difference as percentage of baseline (e.g., 40% change)
- **Retention Ratio**: Fraction of gradient strength remaining after ablation (e.g., 0.60 = 60% retained)
- **Reduction Ratio**: Fraction of gradient strength lost due to ablation (e.g., 0.40 = 40% lost)

**Why two types of ratios?**  
- **Baseline contribution** shows state encoder's share of total gradient flow (no ablation needed)
- **Reduction ratio** shows what happens when state encoder is removed (requires ablation)
- Together they give complete picture: "How much flows through?" + "What happens without it?"

**Interpretation:**
- High baseline contribution (>0.05) + High reduction ratio (>0.3) → State encoder is critical
- Low baseline contribution (<0.02) + Low reduction ratio (<0.1) → Model is vision/language-dominated

**Why use a script?** The script executes in the `evo1_server` conda environment which has:
- Flash Attention (required for InternVL3-1B)
- transformers==4.57.6 (NOT 5.0.0 - critical for meta tensor compatibility)
- PyTorch 2.5.1+cu121

Regular notebook cells run in Colab's default Python (missing these dependencies).

In [ ]:
# Run complete gradient flow analysis using runner script
# This runs in evo1_server conda environment which has all required dependencies
# (Flash Attention, transformers==4.57.6, PyTorch 2.5.1, etc.)

import os

print('🔬 Running Evo-1 Gradient Flow Analysis')
print('='*60)
print('This script will:')
print('  1. Load Evo-1 model (in evo1_server conda environment)')
print('  2. Attach gradient hooks')
print('  3. Run baseline analysis (normal state)')
print('  4. Run ablation analysis (zeroed state encoder)')
print('  5. Compare gradients and generate verdict')
print('='*60)
print()

# Change to EmbodiedLLM repo directory (where scripts are)
os.chdir('/content/EmbodiedLLM/MultipleHooksStudy')

# Run the analysis script in evo1_server environment
# This ensures all dependencies (Flash Attention, transformers==4.57.6) are correct
print('⏳ Running analysis (takes 2-3 minutes)...')
print()

!conda run -n evo1_server python scripts/run_evo1_gradient_analysis.py --checkpoint metaworld

print('\n✅ Analysis complete!')
print('   Results saved to: /content/evo1_gradient_analysis_results.json')
print('   Run next cell to display detailed results')
print()
print('💡 To analyze LIBERO checkpoint instead, run:')
print('   !conda run -n evo1_server python scripts/run_evo1_gradient_analysis.py --checkpoint libero')

## 3. Display Analysis Results

The previous cell saved results to `/content/evo1_gradient_analysis_results.json`.  
Run this cell to display the detailed findings.

In [ ]:
# Display the results from the gradient analysis
import json
import os
from pathlib import Path

results_file = '/content/evo1_gradient_analysis_results.json'

if not os.path.exists(results_file):
    print('❌ Results file not found!')
    print(f'   Expected: {results_file}')
    print('   Please run the previous cell (Cell 10) first')
else:
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print('\n' + '='*60)
    print('EVO-1 GRADIENT FLOW ANALYSIS RESULTS')
    print('='*60)
    
    print(f"\n📊 Model Information:")
    print(f"   Model: {results['model']}")
    print(f"   Checkpoint: {results['checkpoint']}")
    print(f"   Device: {results['device']}")
    print(f"   State Encoder: {results['state_encoder']}")
    
    print(f"\n🔬 Gradient Analysis:")
    print(f"   Ablation Method: {results['ablation_method']}")
    print(f"   \n   BASELINE (Normal State):")
    print(f"   Baseline Gradient Norm:  {results['baseline_grad_norm']:.6f}")
    print(f"   Total Model Grad Norm:   {results['total_model_grad_norm']:.6f}")
    print(f"   Baseline Contribution:   {results['baseline_contribution_ratio']:.6f} ({results['baseline_contribution_ratio']*100:.2f}% of total)")
    print(f"   \n   ABLATION (Zeroed State Encoder):")
    print(f"   Ablated Gradient Norm:   {results['ablated_grad_norm']:.6f}")
    print(f"   Absolute Reduction:      {results['gradient_absolute_reduction']:.6f}")
    print(f"   Gradient Change:         {results['gradient_change_pct']:.1f}%")
    print(f"   Retention Ratio:         {results['gradient_retention_ratio']:.3f} ({results['gradient_retention_ratio']*100:.1f}% retained)")
    print(f"   Reduction Ratio:         {results['gradient_reduction_ratio']:.3f} ({results['gradient_reduction_ratio']*100:.1f}% lost)")
    
    print(f"\n📈 Loss Comparison:")
    print(f"   Baseline Loss:  {results['loss_baseline']:.6f}")
    print(f"   Ablated Loss:   {results['loss_ablated']:.6f}")
    
    print(f"\n🎯 VERDICT: {results['verdict']}")
    print(f"\n{results['explanation']}")
    
    print('\n' + '='*60)
    print('Methodology:')
    print('Measured gradient sensitivity when integration_module')
    print('output was replaced with zeros (output ablation).')
    print('='*60)
    
    # Export for later use
    print(f"\n✅ Full results saved to: {results_file}")

✅ Evo-1 hook framework imported


### Advanced Usage: Script Options

The analysis scripts support different checkpoints and custom output locations:

**Analyze LIBERO checkpoint instead of MetaWorld:**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint libero
```

**Custom output location:**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/run_evo1_gradient_analysis.py --checkpoint metaworld --output /content/my_results.json
```

**Load model only (without analysis):**
```python
!conda run -n evo1_server python /content/EmbodiedLLM/MultipleHooksStudy/scripts/load_evo1_model.py --checkpoint metaworld --output /content/model_info.pkl
```

**Why these scripts?**  
- ✅ Execute in correct conda environment (`evo1_server`)
- ✅ Have all required dependencies (Flash Attention, transformers==4.57.6)
- ✅ Version controlled in git repository
- ✅ Reusable across multiple notebooks
- ✅ No need to modify notebook cells for environment compatibility